
# <font color="green">Local arrays on the stack (then call)</font>

## Problem

* Write a function `l2_norm_local(a, b, c)` __in assembly__ that creates a local (stack) array of three `long`s, stores `a, b, c` into it, and returns `l2_norm_long` of it.
* That is, translate the following C function into assembly:
```
long l2_norm_long(long * x);   /* given; computes x[0]*x[0]+x[1]*x[1]+x[2]*x[2] */

long l2_norm_local(long a, long b, long c) {
  long x[3];
  x[0] = a; x[1] = b; x[2] = c;
  return l2_norm_long(x);
}
```
* Remember to keep the stack 16-byte aligned, and to save `x30` (and set up a frame) because you make a call.
* Fill in the skeleton `l2_norm_local.s` (after `// ------- write your answer here -------`).
* The checker `check_l2_norm_local.c` provides `l2_norm_long` and verifies your result. If you see `OK`s and no errors, you are done.

## Hints

* In the *Square norm* problem you were given a pointer. But where do such pointers come from? One common source is a **local array**: an array declared inside a function lives on the stack, and its address can be passed to other functions.
* No function call is needed to allocate it --- the function simply reserves extra space on the stack (by subtracting from `sp`) and the array occupies that space.
* The *Observe* cells below contain `demo_local`, which declares a local array, fills it, and passes it to another function. Compile it and observe:
  * stack space is reserved with something like `sub sp, sp, #...` in the prologue (and released in the epilogue);
  * the address of the local array is formed from `sp` (e.g. `mov x0, sp` or `add x0, sp, #offset`) and passed in `x0`;
  * because the array's address is taken and handed to another function, the values must really live in memory (not just in registers).



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=l2_norm_local.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* A local (stack) array whose address is passed to another function.
   Observe: stack space reserved with `sub sp, sp, #..`; the address of the
   local formed from sp (`mov x0, sp` or `add x0, sp, #off`); the values stored
   into the array before the call; and the stack released afterwards. */
long use_pair(long * p);   /* some opaque function that reads p[0], p[1] */
long demo_local(long a, long b) {
  long t[2];
  t[0] = a;
  t[1] = b;
  return use_pair(t);
}

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `l2_norm_local.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ l2_norm_local.s
	.arch armv8-a
	.file	"l2_norm_local.c"
	.text
	.align	2
	.p2align 4,,11
	.global	l2_norm_local
	.type	l2_norm_local, %function
l2_norm_local:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	.cfi_endproc
.LFE0:
	.size	l2_norm_local, .-l2_norm_local
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `l2_norm_local` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_l2_norm_local.c
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
long l2_norm_local(long a, long b, long c);

/* the opaque function your code calls */
long l2_norm_long(long * x) {
  return x[0] * x[0] + x[1] * x[1] + x[2] * x[2];
}

int main(int argc, char ** argv) {
  assert(argc == 4);
  long a = atol(argv[1]);
  long b = atol(argv[2]);
  long c = atol(argv[3]);
  long r = l2_norm_local(a, b, c);
  long rc = a * a + b * b + c * c;
  if (r == rc) { printf("OK %ld %ld\n", r, rc); return 0; }
  else { printf("NG %ld %ld\n", r, rc); return 1; }
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `l2_norm_local.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_l2_norm_local -O3 check_l2_norm_local.c l2_norm_local.s -lm


# 6. Run
* The commands to run the checker are stored in `run.sh`.
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_l2_norm_local 1 2 3
./check_l2_norm_local 3 4 5


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_l2_norm_local -O0 -g check_l2_norm_local.c l2_norm_local.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_l2_norm_local
(gdb) break l2_norm_local
(gdb) run ...        # give the same arguments as in run.sh
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=l2_norm_local.md answer_file=l2_norm_local.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.